# Introduction 

Fix $n \ge 2.$ Recall the **$N \times N \times N$ Rubiks Cube** is a cube with each of its $6$ square faces divided into $N^2$ colored squares known as **stickers**. Each sticker takes on a color from a set of $6$ possible colors. We refer to an **$N$-dimensional cube configuration** as a particular state of the cube with all its $6N^2$ colored stickers. In particular, an **$N$-dimensional solved configuration** is a cube configuration such that each face has $N^2$ stickers are of the same color. We show examples of an arbitrary $N$-dimensional cube configurations and solved configurations.

<table>
<tr>
<td> <img src="https://hlavolam.maweb.eu/images/rubikova%20kostka%202x2x2%201.gif" width="100">
<figcaption> 2D Configuration </figcaption>

<td> <img src="https://hlavolam.maweb.eu/images/rubikova%20kostka%202x2x2%202.gif" width="100">
<figcaption> Solved 2D Configuration </figcaption>

<tr>
<td> <img src="https://hlavolam.maweb.eu/images/rubikova%20kostka%203x3x3%201.gif" width="100">
<figcaption> 3D Configuration </figcaption>

<td> <img src="https://hlavolam.maweb.eu/images/rubikova%20kostka%203x3x3%202.gif" width="100">
<figcaption> Solved 3D Configuration </figcaption>
<tr>
<td> <img src="https://hlavolam.maweb.eu/images/rubikova%20kostka%204x4x4%201.gif" width="100">
<figcaption> 4D Configuration </figcaption>

<td> <img src="https://hlavolam.maweb.eu/images/rubikova%20kostka%204x4x4%202.gif" width="100">
<figcaption> Solved 4D Configuration </figcaption>

</tr>
</table>

Let $\mathcal{C}_N$ be the set of all possible, valid $n$-dimensional cube configurations. If $c,d \in \mathcal{C}_N,$ we can obtain $d$ from $c$ via performing a sequence of clockwise or counterclockwise **face rotations (scrambles)** in succession. 

Given any $c \in \mathcal{C}_N,$ the standard puzzle is to **solve** $c,$ which is to find a sequence of scrambles that transforms $c$ into an $N$-dimensional solved configuration upon performing such a sequence in succession. Let $m_c$ be the minimal number of scrambles needed to solve $c.$ What is **God's Number** for the $N$-dimensional Rubiks cube, which is the number $$M_N= \max_{c \in \mathcal{C}_N} m_c.$$ More informally, what is the maximum number of moves needed to *efficiently* solve the Rubiks cube ? 

Now, begin with a solved $n$-dimensional cube configuration. For each $t \in \mathbb{N},$ let $C_t$ be the discrete random variable defined on the state space $\mathcal{C}_t$ which represents the configuration obtained after performing $t$ scrambles on the solved configuration in succession. What is the probability distribution of $C_t$ ? Is it asymptotically uniform ? More informally, is it possible to "shuffle" the Rubiks Cube ? 


We answer our posed questions using Group Theory and Markov Chain Theory.





In [1]:
import os
import numpy as np
import scipy as sc
import pandas as pd
import matplotlib.pyplot as plt
import copy
import json
from sympy.combinatorics import Permutation
from sympy.combinatorics.named_groups import SymmetricGroup
from matplotlib.patches import Polygon
from mpl_toolkits.mplot3d.art3d import Poly3DCollection,Line3DCollection
from ipywidgets import interact, interactive, fixed, interact_manual,FloatSlider
import ipywidgets as widgets
face_names = ['Front','Left','Up','Back','Right','Down']

# Encoding Cube Configurations

We begin by formally defining a Rubiks cube configuration for our purposes.


An $N$-dimensional cube configuration $c$ is a $6$-tuple of $N \times N$ matrices $$c=\left( \mathbf{F}, \mathbf{L}, \mathbf{U}, \mathbf{B}, \mathbf{R}, \mathbf{D}  \right),$$ where $\mathbf{F}, \mathbf{L}, \mathbf{U}, \mathbf{B}, \mathbf{R}, \mathbf{D}$ are the front, left, up, back, right, and down **face matrices** of the configuration. For each face matrix $\mathbf{M},$ its $(m,n)$-th entry $M^{(m,n)}$ is an integer from $\lbrace 0, \ \dots \ ,6N^2-1 \rbrace,$ representing the value of the $N(m-1)+n$-th sticker of the face. Furthermore, we require the configuration's total $6N^2$ sticker values to be distinct integers from the set $\lbrace 0, \ \dots \ , 6N^2-1 \rbrace.$ 



We define our **$N$-dimensional solved configuration** to be the configuration $\iota_N,$ in which for all $m,n,$ we have $F^{(m,n)}=N(m-1)+n-1, L^{(m,n)}=N^2+F^{(m,n)}, \ \dots \ , D^{(m,n)}=5N^2+D^{(m,n)}.$ In addition, we also define the **sticker color dictionary** for visualization purposes. Each $F^{(m,n)}, L^{(m,n)}, U^{(m,n)}, B^{(m,n)}, R^{(m,n)}, D^{(m,n)}$ of our solved configuration is assigned to colors white, blue, green, yellow, red, and orange. respectively.  

In the code cell below, we output our $\iota_N$ as a dictionary. The keys are the face names and the values are their corresponding $N \times N$ face matrices. We also plot its 3D colored representation and net diagram representation according to its standard color dictionary.



In [2]:
def get_solved_configuration_dict(N):
    face_names=['Front','Left','Up','Back','Right','Down']
    face_matrices=[np.arange(i*N**2,(i+1)*N**2).reshape(N,N) for i in range(6)]
    solved_configuration_dict=dict(zip(face_names,face_matrices))
    return solved_configuration_dict

def get_sticker_colors_dict(N):
    sticker_numerical_labels=range(6*N**2)
    colors=(N**2)*['white'] + (N**2)*['blue']+(N**2)*['green']+(N**2)*['yellow']+(N**2)*['red']+(N**2)*['orange']
    sticker_colors_dict=dict(zip(sticker_numerical_labels,colors))
    return sticker_colors_dict


N=2
print(get_solved_configuration_dict(N))

{'Front': array([[0, 1],
       [2, 3]]), 'Left': array([[4, 5],
       [6, 7]]), 'Up': array([[ 8,  9],
       [10, 11]]), 'Back': array([[12, 13],
       [14, 15]]), 'Right': array([[16, 17],
       [18, 19]]), 'Down': array([[20, 21],
       [22, 23]])}


In [100]:
def plot_configuration_3D(configuration_dict, 
                          sticker_colors_dict=None,
                          show_front_numerical_face_labels=False,
                          show_left_numerical_face_labels=False,
                          show_up_numerical_face_labels=False,
                          show_back_numerical_face_labels=False,
                          show_right_numerical_face_labels=False,
                          show_down_numerical_face_labels=False,
                          title='Configuration', 
                          figsize=(5,5), 
                          alpha=0.5, 
                          linewidth=4.0,
                          view_flat_angle=-45,
                          view_elev_angle=45/2):
    
    N=len(configuration_dict['Front'])
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection='3d')
    
    if sticker_colors_dict is None:
        sticker_colors_dict= get_sticker_colors_dict(N)
        
    front_squares=[np.array([[i,0,N-j], [i+1,0,N-j], [i+1,0,N-j-1], [i,0,N-j-1]]) for j in range(N) for i in range(N)]
    left_squares=[np.array([[0,N-i,N-j],[0,N-i-1,N-j],[0,N-i-1,N-j-1],[0,N-i,N-j-1]]) for j in range(N) for i in range(N)]
    up_squares=[np.array([[i,N-j,N],[i+1,N-j,N],[i+1,N-j-1,N],[i,N-j-1,N]]) for j in range(N) for i in range(N)]

    back_squares=[np.array([[N-i,N,N-j],[N-i-1,N,N-j],[N-i-1,N,N-j-1],[N-i,N,N-j-1] ]) for j in range(N) for i in range(N)]
    right_squares=[np.array([[N,i,N-j],[N,i+1,N-j],[N,i+1,N-j-1],[N,i,N-j-1] ]) for j in range(N) for i in range(N)]
    down_squares=[np.array([[i,j,0],[i+1,j,0],[i+1,j+1,0],[i,j+1,0]]) for j in range(N) for i in range(N)]
    
    

    for i in range(N**2):
        squares=front_squares[i], left_squares[i], up_squares[i], back_squares[i], right_squares[i], down_squares[i]
        labels=[str(configuration_dict[face_name].flatten()[i])+ face_name[0] for face_name in configuration_dict.keys()]
        centers=[np.mean(square,axis=0) for square in squares]

        
        
        if show_front_numerical_face_labels:
            alpha=0
            center,label=centers[0],labels[0]
            ax.text(center[0],center[1],center[-1],s=label,zdir='x',alpha=1.0)
        if show_left_numerical_face_labels:
            alpha=0
            center,label=centers[1],labels[1]
            ax.text(center[0],center[1],center[-1],s=label,zdir='y',alpha=1.0)  
        if show_up_numerical_face_labels:
            alpha=0
            center,label=centers[2],labels[2]
            ax.text(center[0],center[1],center[-1],s=label,zdir='x',alpha=1.0)
        if show_back_numerical_face_labels:
            alpha=0
            center,label=centers[3],labels[3]
            ax.text(center[0],center[1],center[-1],s=label,zdir='x',alpha=1.0)
        if show_right_numerical_face_labels:
            alpha=0
            center,label=centers[4],labels[4]
            ax.text(center[0],center[1],center[-1],s=label,zdir='y',alpha=1.0)
        if show_down_numerical_face_labels:
            alpha=0
            center,label=centers[5],labels[5]
            ax.text(center[0],center[1],center[-1],s=label,zdir='x',alpha=1.0)


        for j in range(6):
            square, label = squares[j], labels[j]
            p=Poly3DCollection([square],facecolor = sticker_colors_dict[int(label[:-1])],edgecolor='black',linewidth=linewidth,alpha=alpha)
            ax.add_collection3d(p)
    
        
            
    ax.set_xlim(0,N)
    ax.set_ylim(0,N)
    ax.set_zlim(0,N)
    
    ax.axis('off')
    ax.set_title(title) 
    ax.view_init(view_elev_angle,view_flat_angle)


def plot_configuration_net_diagram(configuration_dict, 
                                   sticker_colors_dict=None,
                                   show_numerical_labels=True,
                                   figsize=(5,5),
                                   title="Configuration Net Diagram"):
    
    n=len(configuration_dict['Front'])
    fig,ax=plt.subplots(figsize=figsize)

    if sticker_colors_dict is None:
        sticker_colors_dict = get_sticker_colors_dict(N)

    front_squares=[np.array([[i,N-j],[i+1,N-j],[i+1,N-j-1],[i,N-j-1]]) for j in range(N) for i in range(N)]
    left_squares=[np.array([[-N+i,N-j],[-N+i+1,n-j],[-N+i+1,N-j-1],[-N+i,N-j-1]]) for j in range(N) for i in range(N)]
    up_squares=[np.array([[i,2*N-j],[i+1,2*N-j],[i+1,2*N-j-1],[i,2*N-j-1]]) for j in range(N) for i in range(N)]
    back_squares=[np.array([[i,3*N-j],[i+1,3*N-j],[i+1,3*N-j-1],[i,3*N-j-1]]) for j in range(N) for i in range(N)]
    right_squares=[np.array([[N+i,N-j],[N+i+1,N-j],[N+i+1,N-j-1],[N+i,N-j-1]]) for j in range(N) for i in range(N)]
    down_squares=[np.array([[i,-j],[i+1,-j],[i+1,-j-1],[i,-j-1]]) for j in range(N) for i in range(N)]

    for i in range(N**2):
        squares=front_squares[i], left_squares[i], up_squares[i], back_squares[i], right_squares[i], down_squares[i]
        centers=[np.mean(square,axis=0) for square in squares]
        labels=[str(configuration_dict[face_name].flatten()[i])+ face_name[0] 
                for face_name in configuration_dict.keys()]
        
        for j in range(6):
            square, label, center = squares[j], labels[j], centers[j]
            p = Polygon(square, facecolor = sticker_colors_dict[int(label[:-1])],
                        edgecolor='black',
                        linewidth=4.0,
                        alpha=1.0)
            ax.add_patch(p)
            if show_numerical_labels:
                ax.text(center[0]-0.1,center[1]-0.1,s=label,alpha=1.0)
    
    ax.set_xlim(-N,2*N)
    ax.set_ylim(-N,3*N)
    ax.axis('off')
    ax.set_title(title)

        
    

N=10
solved_configuration_dict = get_solved_configuration_dict(N) 


interact(plot_configuration_3D, 
         configuration_dict=fixed(solved_configuration_dict), 
         sticker_colors_dict=fixed(None),
         title=fixed("{}D Solved Configuration".format(N)), 
         figsize=fixed((5,5)), 
         alpha=FloatSlider(min=0,max=1,step=1.0,value=1.0,continuous_update=False), 
         linewidth=fixed(4.0),
         view_flat_angle=FloatSlider(min=-360, max=360, step=45/2,value=-45,continuous_update=False),
         view_elev_angle=FloatSlider(min=-360, max=360, step=45/2,value=45/2,continuous_update=False))


interact(plot_configuration_net_diagram, 
         sticker_colors_dict=fixed(None),
         show_numerical_labels=True,
         configuration_dict=fixed(solved_configuration_dict), 
         title=fixed("{}D Solved Configuration Net Diagram".format(N)), 
         figsize=fixed((20,20)))

interactive(children=(Checkbox(value=False, description='show_front_numerical_face_labels'), Checkbox(value=Fa…

interactive(children=(Checkbox(value=True, description='show_numerical_labels'), Output()), _dom_classes=('wid…

<function __main__.plot_configuration_net_diagram(configuration_dict, sticker_colors_dict=None, show_numerical_labels=True, figsize=(5, 5), title='Configuration Net Diagram')>

# Encoding Face Rotations



Let $\mathcal{C}_N$ be the set of all valid cube configurations containing $\iota_N.$ For each face matrix $\mathbf{M},$ denote the $n$-th row of $\mathbf{M}$ as $M^{(n \ , \ )}$ and the $n$-th column as $M^{( \ , \ n)}.$

For each $n \in \lbrace 1, \ \dots \ , N-1 \rbrace,$ we define the **rotation functions** $\mathbf{f}_n,\mathbf{l}_n,\mathbf{u}_n,: \mathcal{C}_N \rightarrow \mathcal{C}_N,$ in which for each $c=\left( \mathbf{F}, \mathbf{L}, \mathbf{U}, \mathbf{B}, \mathbf{R}, \mathbf{D}  \right) \in \mathcal{C}_N,$ we 

* $\mathbf{m}_1(c)$ replaces $M^{( \ , \ m)}$ with $M^{(N-m \ , \ )},$ for each $\mathbf{m} \in \lbrace \mathbf{f}, \mathbf{l}, \mathbf{u} \rbrace.$ 

* For each $n,$ $\mathbf{f}_n(c)$ replaces $U^{(N-n \ , \ )}$ with $L^{(\ , \ N-n)},$ $R^{( \ , \ n)}$ with $U^{(N-n \ , \ )},$ $D^{(n \ , \ )}$ with $R^{( \ , \ n)},$ and finally $L^{(\ , \ N-n)}$ with $D^{(n \ , \ )}.$

* For each $n,$ $\mathbf{l}_n(c)$ replaces $U^{( \ , \ n )}$ with $B^{(\ , \ N-n)},$ $F^{( \ , \ n)}$ with $U^{( \ , \ n)},$ $D^{( \ , \ n)}$ with $F^{( \ , \ n)},$ and finally $B^{(\ , \ N-n)}$ with $D^{( \ , \ n)}.$

* For each $n,$ $\mathbf{u}_n(c)$ replaces $F^{(n \ , \  )}$ with $R^{(n\ , \ )},$ $L^{(n \ , \ )}$ with $F^{(n \ , \ )},$ $B^{(n \ , \ )}$ with $L^{(n \ , \ )},$ and finally $R^{(n \ , \ )}$ with $B^{(n \ , \ )}.$


It is easy to verify these clockwise rotation functions have inverse functions $\mathbf{f}_n^{-1},\mathbf{l}_n^{-1},\mathbf{u}_n^{-1},$ which represent basic **basic counterclockwise rotation functions**. Furthermore, it is easy to verify $\mathbf{m}_i$ and $\mathbf{m}_j$ are commutative and $(\mathbf{m}_1 \ \circ \ \dots \ \circ \mathbf{m}_{N-1})^{-1}$ is the clockwise rotation function corresponding to the nonadjacent face of $\mathbf{M}$ on the cube.  

Let $$\mathcal{B}_N=\left \lbrace \mathbf{m}_1 \circ \ \dots \ \circ \mathbf{m}_{n}, \ (\mathbf{m}_1 \circ \ \dots \ \circ \mathbf{m}_{n})^2 \ , \left(\mathbf{m}_1 \circ \ \dots \ \circ \mathbf{m}_{n}\right)^{-1}: \mathbf{m} \in \left \lbrace \mathbf{f}, \mathbf{l}, \mathbf{u} \right \rbrace , n \in \left \lbrace 1, \ \dots \ , N-1 \right \rbrace \right \rbrace,$$ and $\langle \mathcal{B}_N \rangle$ denote the subgroup of automorphisms on $\mathcal{C}_N$ generated by $\mathcal{B}_N.$ 

**Theorem**: $\langle \mathcal{B}_N \rangle$  is isomorphic to a subgroup of $S_{6N^2},$ and $\mathcal{C}_N= \left \lbrace \mathbf{b}(\iota_N): \mathbf{b} \in  \langle \mathcal{B}_N \rangle \right \rbrace.$ Therefore, $\mathcal{C}_N$ is in bijective correspondence with a subgroup of $S_{6N^2}.$


For each $\mathbf{b} \in \mathcal{B}_N,$ we identify $\mathbf{b} \in S_{6N^2}$ with the unique permutation of $6N^2$ numerical stickers induced by $\mathbf{b}.$ Therefore, without confusion, $\mathbf{b}$ is now understood to be an element of $S_{6N^2}$. 


In the code cell below, we output the permutations in $\mathcal{B}_N$ written in cyclic notation. 

In [103]:
def rotate_array(M):
    return M[::-1].T

def rotate_clockwise(configuration_dict, move_str):
    N=len(configuration_dict['Front'])
    old_configuration_dict, new_configuration_dict = copy.deepcopy(configuration_dict), copy.deepcopy(configuration_dict)
    old_front, old_left, old_up, old_back, old_right, old_down = [old_configuration_dict[face_name] for face_name in face_names]
    new_front, new_left, new_up, new_back, new_right, new_down = [new_configuration_dict[face_name] for face_name in face_names]
    move_name, slice_num = move_str[0], int(move_str[-1])-1
  
    if slice_num ==0 and move_name =='f': 
        new_front = rotate_array(old_front)
    if slice_num ==0 and move_name =='l':
        new_left = rotate_array(old_left)
    if slice_num ==0 and move_name =='u':
        new_up = rotate_array(old_up)
  

    if move_name =='f':
        new_up[N-slice_num-1]  = old_left[:, N-slice_num-1][::-1]
        new_right[:,slice_num] = old_up[N-slice_num-1]
        new_down[slice_num] = old_right[:,slice_num][::-1]
        new_left[:, N-slice_num-1]= old_down[slice_num]
  
    if move_name =='l':
        new_up[:,slice_num]  = old_back[:, N-slice_num-1][::-1]
        new_front[:, slice_num] = old_up[:,slice_num]
        new_down[:, slice_num] = old_front[:,slice_num]
        new_back[:, N-slice_num-1] = old_down[:, slice_num][::-1]

    if move_name =='u':
        new_front[slice_num]  = old_right[slice_num]
        new_left[slice_num]  = old_front[slice_num]
        new_back[slice_num]  = old_left[slice_num]
        new_right[slice_num]  = old_back[slice_num]

    new_configuration_dict = dict(zip(face_names, [new_front, new_left, new_up, new_back, new_right, new_down]))  
    return new_configuration_dict 

def rotate_counterclockwise(configuration_dict, move_str):
    new_configuration_dict = copy.deepcopy(configuration_dict)
    for i in range(3):
        new_configuration_dict = rotate_clockwise(new_configuration_dict, move_str)
    return new_configuration_dict


def get_configuration_permutation(cube_configuration):
    N = len(cube_configuration['Front'])
    numerical_labels = list(np.hstack([cube_configuration[face_name].flatten() for face_name in face_names]))
    configuration_permutation = Permutation(numerical_labels,size=6*N**2)
    return configuration_permutation


def get_rotation_permutations_dict(N):
    solved_configuration_dict = get_solved_configuration_dict(N)
    rotation_permutations_dict = dict()
    for m in ['f','l','u']:
        clockwise_permutation=Permutation(size = 6*N**2)
        for slice_num in range(1,N):
            clockwise_move_str="".join([m+str(i) for i in range(1,slice_num+1)])
            clockwise_permutation*= get_configuration_permutation(rotate_clockwise(solved_configuration_dict, m+str(slice_num)))
            clockwise_twice_move_str, counterclockwise_move_str="({})^2".format(clockwise_move_str), "({})^-1".format(clockwise_move_str)
            clockwise_twice_permutation, counterclockwise_permutation =(clockwise_permutation)**2, (clockwise_permutation)**3
            rotation_permutations_dict.update({clockwise_move_str : clockwise_permutation})
            rotation_permutations_dict.update({clockwise_twice_move_str : clockwise_twice_permutation})
            rotation_permutations_dict.update({counterclockwise_move_str : counterclockwise_permutation})
            
    return rotation_permutations_dict

N=3
rotation_permutations_dict=get_rotation_permutations_dict(N)
print(rotation_permutations_dict)
# B=SymmetricGroup(6*N**2).subgroup(list(rotation_permutations_dict.values()))
# B.order()

{'f1': Permutation(53)(0, 6, 8, 2)(1, 3, 7, 5)(11, 45, 42, 26)(14, 46, 39, 25)(17, 47, 36, 24), '(f1)^2': Permutation(53)(0, 8)(1, 7)(2, 6)(3, 5)(11, 42)(14, 39)(17, 36)(24, 47)(25, 46)(26, 45), '(f1)^-1': Permutation(53)(0, 2, 8, 6)(1, 5, 7, 3)(11, 26, 42, 45)(14, 25, 39, 46)(17, 24, 36, 47), 'f1f2': Permutation(53)(0, 6, 8, 2)(1, 3, 7, 5)(10, 48, 43, 23)(11, 45, 42, 26)(13, 49, 40, 22)(14, 46, 39, 25)(16, 50, 37, 21)(17, 47, 36, 24), '(f1f2)^2': Permutation(53)(0, 8)(1, 7)(2, 6)(3, 5)(10, 43)(11, 42)(13, 40)(14, 39)(16, 37)(17, 36)(21, 50)(22, 49)(23, 48)(24, 47)(25, 46)(26, 45), '(f1f2)^-1': Permutation(53)(0, 2, 8, 6)(1, 5, 7, 3)(10, 23, 43, 48)(11, 26, 42, 45)(13, 22, 40, 49)(14, 25, 39, 46)(16, 21, 37, 50)(17, 24, 36, 47), 'l1': Permutation(53)(0, 18, 35, 45)(3, 21, 32, 48)(6, 24, 29, 51)(9, 15, 17, 11)(10, 12, 16, 14), '(l1)^2': Permutation(53)(0, 35)(3, 32)(6, 29)(9, 17)(10, 16)(11, 15)(12, 14)(18, 45)(21, 48)(24, 51), '(l1)^-1': Permutation(53)(0, 45, 35, 18)(3, 48, 32, 21)(6,

# Determining God's Number with A Devil's Algorithm

We now algorithmically determine **God's Number** for the $N$-dimensional Rubiks cube, which is the number $$M_N= \max_{c \in \mathcal{C}_N} m_c,$$ where $m_c$ is the minimal number of face rotations needed to bring $c$ to $\iota_N.$  More informally, $M_N$ is the maximum number of moves needed to *efficiently* solve the Rubiks cube. We determine this using a "Devil's Algorithm."

* Let $\mathcal{H}_1=\lbrace \textbf{Id}_{6N^2} \rbrace.$ 
* For each $k \in \mathbb{N},$  we let $$\mathcal{H}_{k+1}= \lbrace  \mathbf{a}\circ \mathbf{b}: \mathbf{a} \in \mathcal{H}_k, \mathbf{b} \in \mathcal{B}_N  \rbrace.$$
* We stop once $\mathcal{H}_k= \langle \mathcal{B}_N \rangle.$ 

**Theorem** : The smallest such $k$ for which $\mathcal{H}_k= \langle \mathcal{B}_N \rangle$ is precisely $M_N.$



In [104]:
class RubiksCubeMarkovChain:
    def __init__(self, N):
        self.N=N
        self.current_permutations_dict = self.get_current_permutations_dict()
        self.devils_algorithm_iterations_dict = self.get_devils_algorithm_iterations_dict()
        self.rotation_permutations = list(get_rotation_permutations_dict(self.N).values())
        self.current_permutations = self.get_current_permutations()
        self.current_iteration_permutations = self.current_permutations 
        
    def get_current_permutations_dict(self):
        directory="{}D Chain".format(self.N)
        try:
            os.makedirs(directory)
        except:
            pass
        file_path="{}/Current_Permutations.json".format(directory)
        current_permutations_dict = {0: Permutation(size=6*self.N**2).cyclic_form}
        if os.path.exists(file_path):
            with open(file_path, 'r') as fp:
                print("LOADING CURRENT PERMUTATIONS DICTIONARY ...")
                current_permutations_dict = json.load(fp)
                current_permutations_dict = dict(enumerate(current_permutations_dict.values()))
        return current_permutations_dict

    def get_devils_algorithm_iterations_dict(self):
        directory="{}D Chain".format(self.N)
        file_path="{}/Devils_Algorithm_Iterations.json".format(directory)
        devils_algorithm_iterations_dict = {self.N : {0 : 1}}
        if os.path.exists(file_path):
            with open(file_path, 'r') as fp:
                devils_algorithm_iterations_dict = json.load(fp)
                int_keys= [int(key) for key in devils_algorithm_iterations_dict.keys()]
                vals=devils_algorithm_iterations_dict.values()
                devils_algorithm_iterations_dict = dict(zip(int_keys, vals))
        return devils_algorithm_iterations_dict



    def get_current_permutations(self):
        current_cyclic_forms = self.current_permutations_dict.values()
        print("CONVERTING CURRENT CYCLIC FORMS TO PERMUTATIONS ...")
        current_permutations = {Permutation(current_cyclic_form, size=6*self.N**2) 
                                for current_cyclic_form in current_cyclic_forms}
        return current_permutations

    def update_current_permutations_dict(self):
        directory="{}D Chain".format(self.N)
        file_path="{}/Current_Permutations.json".format(directory)
        print("OBTAINING NEW PERMUTATIONS ...")
        

        
        new_permutations = {current_iteration_permutation * rotation_permutation 
                            for current_iteration_permutation in self.current_iteration_permutations
                            for rotation_permutation in self.rotation_permutations}
            

            
        self.current_iteration_permutations = new_permutations - self.current_permutations 
        
        if len(self.current_iteration_permutations) > 0:
            new_keys = range(len(self.current_permutations), len(self.current_permutations) + len(self.current_iteration_permutations))
            print("CONVERTING UNIQUE NEW PERMUTATIONS TO CYCLIC FORMS ...")
            new_unique_cyclic_forms = [new_permutation.cyclic_form 
                                       for new_permutation in self.current_iteration_permutations]
            
            new_dict=dict(zip(new_keys, new_unique_cyclic_forms))
            
            self.current_permutations_dict.update(new_dict)
            self.current_permutations = self.current_permutations.union(self.current_iteration_permutations)
            
        
            with open(file_path, 'w') as fp:
                print("UPDATING CURRENT PERMUTATIONS DICTIONARY ...")
                json.dump(self.current_permutations_dict, fp)

        print("DONE !!!")
                
    def update_devils_algorithm_iterations_dict(self):
        directory="{}D Chain".format(self.N)
        file_path="{}/Devils_Algorithm_Iterations.json".format(directory)
        iteration_number =len(self.devils_algorithm_iterations_dict[self.N])
        self.devils_algorithm_iterations_dict[self.N].update({iteration_number : len(self.current_permutations_dict)})
        with open(file_path ,'w') as fp:
            json.dump(self.devils_algorithm_iterations_dict, fp)
            
    def update_chain(self):
        self.update_current_permutations_dict()
        self.update_devils_algorithm_iterations_dict()


N = 2
RMC = RubiksCubeMarkovChain(N)

LOADING CURRENT PERMUTATIONS DICTIONARY ...
CONVERTING CURRENT CYCLIC FORMS TO PERMUTATIONS ...


In [98]:
RMC.update_chain()
RMC.devils_algorithm_iterations_dict

OBTAINING NEW PERMUTATIONS ...
CONVERTING UNIQUE NEW PERMUTATIONS TO CYCLIC FORMS ...
UPDATING CURRENT PERMUTATIONS DICTIONARY ...
DONE !!!


{10: {0: 1, 1: 82, 2: 5428}}

# References

* God's Number Tabulated: http://oeis.org/A080656

* Images : https://hlavolam.maweb.eu/

* Devil's Algorithm: http://anttila.ca/michael/devilsalgorithm/